# EngageMetrics Workforce Analytics

## Overview 
A leading employee engagement analytics company, has just received datasets from multiple sources that need to be consolidated for an urgent executive presentation. The data includes employee feedback from regional offices in CSV format, educational achievement records in Excel spreadsheets, and compensation benchmarks from an external API. The leadership team needs insights on employee engagement trends.

Two different EngageMetrics data sources and an external API from Wikipedia:
- <b>education_data.xlsx:</b> Contains employee educational backgrounds including graduation years and fields of study
- <b>employee_insights.csv:</b> Contains employee performance metrics, satisfaction scores, and work-related information
- <b>Wikipedia REST API:</b> Income statistics in the United States

## Data Loading and Initial Exploration

Import required libraries and load datasets

In [1]:
# Import required libraries

%pip install openpyxl

import pandas as pd
import requests
import json
from datetime import datetime


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Import the quarterly employee engagement feedback data from all regions.  Import and examine the CSV data.

In [2]:
# Import the employee insights data
employee_df = pd.read_csv('../data/employee_insights.csv')

# Display the first few rows
print("\nEmployee Data:")
employee_df.head(25)


Employee Data:


,employee_id,age,salary,promotion_eligible,last_training_date,department,work_experience,projects_completed,hours_worked_weekly,work_mode,last_promotion_date,satisfaction_score,overtime_hours
0,E0001,54.0,NaN,NaN,15/08/2023,HR,NaN,14.0,NaN,remote work,2022-05-10,NaN,8.4
1,E0002,NaN,$64761,N,15/08/2023,NaN,1 years,NaN,53.3,HYBRID,05-10-2022,NaN,8.1
2,E0003,54.0,NaN,N,15/08/2023,Marketing,8,6.0,32.6,Hybrid,10/05/2022,10.0,5.2
3,E0004,NaN,NaN,No,NaN,NaN,16,1.0,37.8,Remote,05-10-2022,5.0,NaN
4,E0005,29.0,$61486,Y,15/08/2023,NaN,NaN,1.0,53.3,Hybrid,2022-05-10,NaN,0.3
5,E0006,NaN,93128,No,15/08/2023,HR,6 years,12.0,35.6,HYBRID,10/05/2022,3.0,9.7
6,E0007,NaN,$115377,NaN,2023-08-15,Finance,11 years,13.0,33.7,HYBRID,05-10-2022,7.0,1.8
7,E0008,NaN,$51543,NaN,15/08/2023,Finance,NaN,NaN,40.8,HYBRID,2022-05-10,NaN,NaN
8,E0009,NaN,68755,N,NaN,NaN,2,NaN,34.3,remote work,05-10-2022,2.0,2.3
9,E0010,37.0,NaN,Y,2023-08-15,Finance,NaN,NaN,57.9,NaN,NaN,2.0,5.7


#### Handle Missing Values

First check for missing values.

In [4]:
employee_df.isnull().sum()

employee_id             0
age                    56
salary                 37
promotion_eligible     16
last_training_date     29
department             15
work_experience        29
projects_completed     52
hours_worked_weekly    33
work_mode              16
last_promotion_date    26
satisfaction_score     39
overtime_hours         30
dtype: int64

Replace missing values based on the data type.

In [5]:
# Identify the numeric columns in the employee_df
numeric_cols = employee_df.select_dtypes(include='number')
print(numeric_cols)
print('Numeric columns are :', numeric_cols.columns)

     age  projects_completed  hours_worked_weekly  satisfaction_score  \
0   54.0                14.0                  NaN                 NaN   
1    NaN                 NaN                 53.3                 NaN   
2   54.0                 6.0                 32.6                10.0   
3    NaN                 1.0                 37.8                 5.0   
4   29.0                 1.0                 53.3                 NaN   
..   ...                 ...                  ...                 ...   
95  41.0                10.0                 34.0                 9.0   
96   NaN                 NaN                 43.3                 6.0   
97  26.0                 0.0                  NaN                 6.0   
98   NaN                 NaN                 53.3                 1.0   
99   NaN                 NaN                  NaN                 6.0   

    overtime_hours  
0              8.4  
1              8.1  
2              5.2  
3              NaN  
4              0.3

In [6]:
#  Fill missing numeric values with median
employee_df[numeric_cols.columns] = numeric_cols.fillna(numeric_cols.median())

In [7]:
# Columns with date info
date_columns = ['last_training_date', 'last_promotion_date']
print(employee_df['last_training_date'].dtypes)
print(employee_df['last_training_date'].dtypes)

# Identify the categorical columns in the employee_df, excluding 
# the date columns in case they are of object or string type
cat_cols = employee_df.select_dtypes(include=['object','string']).columns.difference(date_columns)
print('Categorical columns are :', cat_cols)

str
str
Categorical columns are : Index(['department', 'employee_id', 'promotion_eligible', 'salary',
       'work_experience', 'work_mode'],
      dtype='str')


In [8]:
# Fill missing categorical values with mode
employee_df[cat_cols] = employee_df[cat_cols].apply(lambda x: x.fillna(x.mode()[0]))
employee_df[cat_cols].head(10)

,department,employee_id,promotion_eligible,salary,work_experience,work_mode
0,HR,E0001,Y,$104328,16 years,remote work
1,Finance,E0002,N,$64761,1 years,HYBRID
2,Marketing,E0003,N,$104328,8,Hybrid
3,Finance,E0004,No,$104328,16,Remote
4,Finance,E0005,Y,$61486,16 years,Hybrid
5,HR,E0006,No,93128,6 years,HYBRID
6,Finance,E0007,Y,$115377,11 years,HYBRID
7,Finance,E0008,Y,$51543,16 years,HYBRID
8,Finance,E0009,N,68755,2,remote work
9,Finance,E0010,Y,$104328,16 years,HYBRID


Fill missing date values with the last known valid value.

In [9]:
employee_df[['last_training_date', 'last_promotion_date']] = employee_df[['last_training_date', 'last_promotion_date']].ffill()

Make sure there are no missing values left in the data set.

In [10]:
employee_df.isnull().sum()

employee_id            0
age                    0
salary                 0
promotion_eligible     0
last_training_date     0
department             0
work_experience        0
projects_completed     0
hours_worked_weekly    0
work_mode              0
last_promotion_date    0
satisfaction_score     0
overtime_hours         0
dtype: int64

#### Fix Data Types and Standardize Formats

##### Date Data

Check the data types for date data.

In [11]:
print(employee_df[date_columns].head(10))

  last_training_date last_promotion_date
0         15/08/2023          2022-05-10
1         15/08/2023          05-10-2022
2         15/08/2023          10/05/2022
3         15/08/2023          05-10-2022
4         15/08/2023          2022-05-10
5         15/08/2023          10/05/2022
6         2023-08-15          05-10-2022
7         15/08/2023          2022-05-10
8         15/08/2023          05-10-2022
9         2023-08-15          05-10-2022


Convert the dates that are in the datetime format (YYYY-MM-DD) to `datetime` data type and also mark the ones that cannot convert.

In [12]:
# Convert the date columns to datetime, identifying inconsistent date formats 
employee_df[date_columns] = employee_df[date_columns].apply(pd.to_datetime, format='mixed', errors='coerce')

Identify the rows where `datetime` conversion didn't work.

In [13]:
# Identify the rows where the conversion failed
inconsistent_rows = employee_df[date_columns].isna()

print("Rows with inconsistent or unparseable formats:")
print(inconsistent_rows)

Rows with inconsistent or unparseable formats:
    last_training_date  last_promotion_date
0                False                False
1                False                False
2                False                False
3                False                False
4                False                False
..                 ...                  ...
95               False                False
96               False                False
97               False                False
98               False                False
99               False                False

[100 rows x 2 columns]


Make sure all date data is in the `datetime` format.

In [14]:
print("\nAfter Conversion:")
print(employee_df[date_columns].head(20))        


After Conversion:
   last_training_date last_promotion_date
0          2023-08-15          2022-05-10
1          2023-08-15          2022-05-10
2          2023-08-15          2022-10-05
3          2023-08-15          2022-05-10
4          2023-08-15          2022-05-10
5          2023-08-15          2022-10-05
6          2023-08-15          2022-05-10
7          2023-08-15          2022-05-10
8          2023-08-15          2022-05-10
9          2023-08-15          2022-05-10
10         2023-08-15          2022-05-10
11         2023-08-15          2022-05-10
12         2023-08-15          2022-05-10
13         2023-08-15          2022-05-10
14         2023-08-15          2022-10-05
15         2023-08-15          2022-10-05
16         2023-08-15          2022-05-10
17         2023-08-15          2022-05-10
18         2023-08-15          2022-05-10
19         2023-08-15          2022-05-10


##### Salary Data

In [15]:
# Examine current salary format
print("Salary values sample:")
print(employee_df['salary'].head())

Salary values sample:
0    $104328
1     $64761
2    $104328
3    $104328
4     $61486
Name: salary, dtype: str


Clean the salary column by removing currency symbols and converting to float.

In [16]:
employee_df['salary'] = employee_df['salary'].replace(r'[\$,]', '', regex=True).astype(float)

Verify the salary data type.

In [17]:
print(employee_df['salary'].dtype)
print(employee_df['salary'].head())

float64
0    104328.0
1     64761.0
2    104328.0
3    104328.0
4     61486.0
Name: salary, dtype: float64


##### Work Mode Data

Examine the work_mode column values.

In [18]:
# Show current state    
print("Original Work Mode Names:")
print(employee_df['work_mode'].unique())

Original Work Mode Names:
<StringArray>
['remote work', 'HYBRID', 'Hybrid', 'Remote', 'On-site']
Length: 5, dtype: str


Standardize the `work_mode` column values to be consistent case and format.

In [19]:
employee_df['work_mode'] = employee_df['work_mode'].str.strip().str.lower()     
print("\nCleaned Work Mode Names:")
print(employee_df['work_mode'].unique())


Cleaned Work Mode Names:
<StringArray>
['remote work', 'hybrid', 'remote', 'on-site']
Length: 4, dtype: str


#### Department Names

In [20]:
# Show current state    
print("Original Department Names:")
print(employee_df['department'].unique())

Original Department Names:
<StringArray>
['HR', 'Finance', 'Marketing', 'engineering', 'Engineering', 'finance']
Length: 6, dtype: str


Finance and engineering departments are written with two different capitalization. Make the department names consistent format.

In [21]:
employee_df['department'] = employee_df['department'].str.strip().str.title()
print("\nCleaned Department Names:")
print(employee_df['department'].unique())


Cleaned Department Names:
<StringArray>
['Hr', 'Finance', 'Marketing', 'Engineering']
Length: 4, dtype: str


## Employee Education Data

Import and examine the talent development team's educational background data:

In [22]:
# Import the educational background data
education_df = pd.read_excel('../data/education_data.xlsx')

print("Education Data:")
print(education_df.head())

Education Data:
      ID  graduation_year   educational_background
0  E0001             2011               Psychology
1  E0002             1995             Architecture
2  E0003             2007  Business Administration
3  E0004             2000  Business Administration
4  E0005             1991                 Medicine


In [26]:
# Check for inconsistencies in educational_background column  
print("Original Academic Background Names:")
print(education_df['educational_background'].unique())

Original Academic Background Names:
<StringArray>
[             'Psychology',            'Architecture',
 'Business Administration',                'Medicine',
              'Statistics',             'Mathematics',
              'Philosophy',        'Computer Science',
               'Chemistry',             'Linguistics',
             'Engineering',                     'Law',
                 'Biology',                 'Physics',
               'Economics',       'Political Science']
Length: 16, dtype: str


There is no inconsistent formatting in the educational background names.

In [27]:
# Check if there are any missing values in the education_df dataset.
education_df.isnull().sum()

ID                        0
graduation_year           0
educational_background    0
dtype: int64

The data set has no missing values.

## Merge Education and Employee Data Sets

In [28]:
# Merge datasets on employee_id
merged_df = pd.merge(
    employee_df,
    education_df,
    left_on='employee_id',
    right_on='ID',
    how='inner'
)

merged_df.head(10)

,employee_id,age,salary,promotion_eligible,last_training_date,department,work_experience,projects_completed,hours_worked_weekly,work_mode,last_promotion_date,satisfaction_score,overtime_hours,ID,graduation_year,educational_background
0,E0001,54.0,104328.0,Y,2023-08-15,Hr,16 years,14.0,43.3,remote work,2022-05-10,6.0,8.40,E0001,2011,Psychology
1,E0002,41.0,64761.0,N,2023-08-15,Finance,1 years,8.0,53.3,hybrid,2022-05-10,6.0,8.10,E0002,1995,Architecture
2,E0003,54.0,104328.0,N,2023-08-15,Marketing,8,6.0,32.6,hybrid,2022-10-05,10.0,5.20,E0003,2007,Business Administration
3,E0004,41.0,104328.0,No,2023-08-15,Finance,16,1.0,37.8,remote,2022-05-10,5.0,6.15,E0004,2000,Business Administration
4,E0005,29.0,61486.0,Y,2023-08-15,Finance,16 years,1.0,53.3,hybrid,2022-05-10,6.0,0.30,E0005,1991,Medicine
5,E0006,41.0,93128.0,No,2023-08-15,Hr,6 years,12.0,35.6,hybrid,2022-10-05,3.0,9.70,E0006,2019,Statistics
6,E0007,41.0,115377.0,Y,2023-08-15,Finance,11 years,13.0,33.7,hybrid,2022-05-10,7.0,1.80,E0007,1998,Mathematics
7,E0008,41.0,51543.0,Y,2023-08-15,Finance,16 years,8.0,40.8,hybrid,2022-05-10,6.0,6.15,E0008,2009,Philosophy
8,E0009,41.0,68755.0,N,2023-08-15,Finance,2,8.0,34.3,remote work,2022-05-10,2.0,2.30,E0009,2007,Business Administration
9,E0010,37.0,104328.0,Y,2023-08-15,Finance,16 years,8.0,57.9,hybrid,2022-05-10,2.0,5.70,E0010,1990,Computer Science


The cleaned and merged data is in the new dataframe, `merged_df`. Data is ready to be used in further analysis.

In [29]:
#Renaming the merged dataset for clarity.
employee_profiles_df=merged_df

#Save clean data into a new CSV file for future use
employee_profiles_df.to_csv('../data/employee_profiles_cleaned.csv', index=False)